In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import plotly.express as px
import matplotlib.pyplot as plt

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/earthquakes-in-indonesia/katalog_gempa.csv
/kaggle/input/earthquakes-in-indonesia/katalog_gempa_v2.tsv


In [2]:
df = pd.read_csv("/kaggle/input/earthquakes-in-indonesia/katalog_gempa.csv")
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92887 entries, 0 to 92886
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   tgl      92887 non-null  object 
 1   ot       92887 non-null  object 
 2   lat      92887 non-null  float64
 3   lon      92887 non-null  float64
 4   depth    92887 non-null  int64  
 5   mag      92887 non-null  float64
 6   remark   92887 non-null  object 
 7   strike1  2735 non-null   float64
 8   dip1     2735 non-null   float64
 9   rake1    2735 non-null   float64
 10  strike2  2735 non-null   float64
 11  dip2     2735 non-null   float64
 12  rake2    2735 non-null   float64
dtypes: float64(9), int64(1), object(3)
memory usage: 9.2+ MB


,tgl,ot,lat,lon,depth,mag,remark,strike1,dip1,rake1,strike2,dip2,rake2
0,2008/11/01,21:02:43.058,-9.18,119.06,10,4.9,Sumba Region - Indonesia,NaN,NaN,NaN,NaN,NaN,NaN
1,2008/11/01,20:58:50.248,-6.55,129.64,10,4.6,Banda Sea,NaN,NaN,NaN,NaN,NaN,NaN
2,2008/11/01,17:43:12.941,-7.01,106.63,121,3.7,Java - Indonesia,NaN,NaN,NaN,NaN,NaN,NaN
3,2008/11/01,16:24:14.755,-3.30,127.85,10,3.2,Seram - Indonesia,NaN,NaN,NaN,NaN,NaN,NaN
4,2008/11/01,16:20:37.327,-6.41,129.54,70,4.3,Banda Sea,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# delete unnecessary columns, unknown data
df.drop(columns=["strike1", "dip1", "rake1", "strike2", "dip2", "rake2"], inplace=True)
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92887 entries, 0 to 92886
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   tgl     92887 non-null  object 
 1   ot      92887 non-null  object 
 2   lat     92887 non-null  float64
 3   lon     92887 non-null  float64
 4   depth   92887 non-null  int64  
 5   mag     92887 non-null  float64
 6   remark  92887 non-null  object 
dtypes: float64(3), int64(1), object(3)
memory usage: 5.0+ MB


,tgl,ot,lat,lon,depth,mag,remark
0,2008/11/01,21:02:43.058,-9.18,119.06,10,4.9,Sumba Region - Indonesia
1,2008/11/01,20:58:50.248,-6.55,129.64,10,4.6,Banda Sea
2,2008/11/01,17:43:12.941,-7.01,106.63,121,3.7,Java - Indonesia
3,2008/11/01,16:24:14.755,-3.30,127.85,10,3.2,Seram - Indonesia
4,2008/11/01,16:20:37.327,-6.41,129.54,70,4.3,Banda Sea


In [4]:
# change obj to date
df["tgl"] = pd.to_datetime(df["tgl"])
df["year"] = df["tgl"].dt.year
df.head()

,tgl,ot,lat,lon,depth,mag,remark,year
0,2008-11-01,21:02:43.058,-9.18,119.06,10,4.9,Sumba Region - Indonesia,2008
1,2008-11-01,20:58:50.248,-6.55,129.64,10,4.6,Banda Sea,2008
2,2008-11-01,17:43:12.941,-7.01,106.63,121,3.7,Java - Indonesia,2008
3,2008-11-01,16:24:14.755,-3.30,127.85,10,3.2,Seram - Indonesia,2008
4,2008-11-01,16:20:37.327,-6.41,129.54,70,4.3,Banda Sea,2008


In [5]:
df_filtered = df[(df['year'] > 2008) & (df['year'] < 2023)]
df_countyear = df_filtered.groupby('year').agg(total_intens=('year', 'count')).reset_index()
df_countyear.head()

,year,total_intens
0,2009,3534
1,2010,3437
2,2011,3948
3,2012,2435
4,2013,2749


In [6]:
fig = px.scatter_mapbox(
    df_filtered,  # Our DataFrame
    lat="lat",
    lon="lon",
    color="mag",
    center={"lat": -6.17, "lon": 106.82},
    zoom=5,
    width=600,
    height=600,
    hover_data=["mag"],
    animation_frame="year",
)

fig.update_layout(
    mapbox_style="open-street-map",
    updatemenus=[{
        "buttons": [
            {
                "args": [None, {"frame": {"duration": 500, "redraw": True}, "fromcurrent": True}],
                "label": "Play",
                "method": "animate"
            },
            {
                "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
                "label": "Pause",
                "method": "animate"
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "steps": [{
            "args": [
                [year],
                {"frame": {"duration": 300, "redraw": True}, "mode": "immediate", "transition": {"duration": 300}}
            ],
            "label": str(year),
            "method": "animate"
        } for year in df_filtered['year'].unique()],
        "currentvalue": {"font": {"size": 20}, "prefix": "Year: ", "visible": True, "xanchor": "center"},
        "transition": {"duration": 300},
    }]
)

fig.show()

In [7]:
# count of earthquakes in each years
px.bar(
    df_countyear,
    x="year",
    y="total_intens"
)

In [8]:
# mean of earthquakes in each years

df_mean = df_filtered.groupby('year').agg(mean_mag=('mag', 'mean')).reset_index()

px.bar(
    df_mean,
    x="year",
    y="mean_mag"
)

Kesimpulan / Conclusion

Bencana gempa sering terjadi di seluruh wilayah indonesia kecuali kalimantan (borneo) dengan kejadian yang cenderung meningkat setiap tahunnya dengan kekuatan gempa sekitar 3.4 - 4.2 skala richter

(Earthquake disasters often occur throughout Indonesia except Kalimantan (Borneo) with incidents tending to increase every year, magnitude's around 3.4 - 4.2)